# Review 2 — K-Means Clustering

**Dataset:** Online Retail Excel. Create customer-level RFM features, choose candidate k with the Elbow curve, fit K-Means, report all required metrics, and visualize clusters with PCA and t-SNE.

## 1. Load, audit, clean and create RFM features

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans,AgglomerativeClustering
from sklearn.metrics import silhouette_score,davies_bouldin_score,calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import linkage,dendrogram
import joblib
ROOT = Path.cwd().resolve()
while not (ROOT / 'data' / 'Online Retail.xlsx').is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError('Dataset not found: data/Online Retail.xlsx')
    ROOT = ROOT.parent
DATA=ROOT/'data'/'Online Retail.xlsx';RESULTS=ROOT/'results'/'clustering';FIGURES=ROOT/'figures'/'clustering';MODELS=ROOT/'models'/'clustering'
for p in [RESULTS,FIGURES,MODELS]:p.mkdir(parents=True,exist_ok=True)
raw=pd.read_excel(DATA);print('Raw shape:',raw.shape);display(raw.isna().sum().sort_values(ascending=False).to_frame('missing_count'))
df=raw.copy();df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'],errors='coerce')
for c in ['CustomerID','Quantity','UnitPrice']:df[c]=pd.to_numeric(df[c],errors='coerce')
df=df.dropna(subset=['CustomerID','InvoiceDate','Quantity','UnitPrice']);df=df[~df['InvoiceNo'].astype(str).str.startswith('C')];df=df[(df['Quantity']>0)&(df['UnitPrice']>0)].drop_duplicates();df['Revenue']=df['Quantity']*df['UnitPrice']
snapshot=df['InvoiceDate'].max()+pd.Timedelta(days=1)
rfm=df.groupby('CustomerID').agg(last_purchase=('InvoiceDate','max'),frequency=('InvoiceNo','nunique'),monetary=('Revenue','sum'),quantity=('Quantity','sum'))
rfm['recency']=(snapshot-rfm['last_purchase']).dt.days;rfm=rfm.drop(columns='last_purchase')[['recency','frequency','monetary','quantity']]
for c in ['frequency','monetary','quantity']:rfm[c]=np.log1p(rfm[c].clip(lower=0))
rfm=rfm.replace([np.inf,-np.inf],np.nan).dropna();rfm.to_csv(RESULTS/'online_retail_customer_rfm.csv',index=True,index_label='CustomerID');print('Customer-level RFM shape:',rfm.shape);display(rfm.head())
scaler=StandardScaler();X_scaled=scaler.fit_transform(rfm)

## 2. Elbow curve

In [ ]:
ks=list(range(2,11));inertias=[]
for k in ks:inertias.append(KMeans(n_clusters=k,random_state=42,n_init=10).fit(X_scaled).inertia_)
elbow=pd.DataFrame({'k':ks,'inertia':inertias});display(elbow);elbow.to_csv(RESULTS/'kmeans_elbow_values.csv',index=False)
fig,ax=plt.subplots(figsize=(7,4));ax.plot(ks,inertias,marker='o');ax.set_title('K-Means Elbow Curve');ax.set_xlabel('Number of clusters (k)');ax.set_ylabel('Inertia');ax.grid(alpha=.3);fig.tight_layout();fig.savefig(FIGURES/'kmeans_elbow_curve.png',dpi=150);plt.show()

## 3. Fit K-Means and evaluate metrics

In [ ]:
K=4 # Initial candidate; confirm using elbow and metrics before final presentation.
kmeans=KMeans(n_clusters=K,random_state=42,n_init=10);labels=kmeans.fit_predict(X_scaled)
metrics=pd.DataFrame([{'algorithm':'K-Means','n_clusters':len(np.unique(labels)),'silhouette':silhouette_score(X_scaled,labels),'davies_bouldin':davies_bouldin_score(X_scaled,labels),'calinski_harabasz':calinski_harabasz_score(X_scaled,labels)}]);display(metrics.round(4));metrics.to_csv(RESULTS/'kmeans_metrics.csv',index=False)
out=rfm.copy();out['kmeans_cluster']=labels;out.to_csv(RESULTS/'kmeans_customer_labels.csv',index=True,index_label='CustomerID');joblib.dump({'model':kmeans,'scaler':scaler},MODELS/'kmeans.joblib')

## 4. PCA visualization

In [ ]:
pca=PCA(n_components=2,random_state=42);Xp=pca.fit_transform(X_scaled);fig,ax=plt.subplots(figsize=(7,5));sc=ax.scatter(Xp[:,0],Xp[:,1],c=labels,s=10,alpha=.65,cmap='tab10');ax.set_title('K-Means Clusters — PCA 2D');ax.set_xlabel('Principal Component 1');ax.set_ylabel('Principal Component 2');fig.colorbar(sc,ax=ax,label='Cluster');fig.tight_layout();fig.savefig(FIGURES/'kmeans_pca.png',dpi=150);plt.show();print('Explained variance ratio:',pca.explained_variance_ratio_)

## 5. t-SNE visualization

In [ ]:
perplexity=min(30,max(5,(len(X_scaled)-1)//3));Xt=TSNE(n_components=2,random_state=42,init='pca',learning_rate='auto',perplexity=perplexity).fit_transform(X_scaled);fig,ax=plt.subplots(figsize=(7,5));sc=ax.scatter(Xt[:,0],Xt[:,1],c=labels,s=10,alpha=.65,cmap='tab10');ax.set_title('K-Means Clusters — t-SNE 2D');ax.set_xlabel('t-SNE 1');ax.set_ylabel('t-SNE 2');fig.colorbar(sc,ax=ax,label='Cluster');fig.tight_layout();fig.savefig(FIGURES/'kmeans_tsne.png',dpi=150);plt.show()